In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import ray
from tqdm.auto import tqdm
from functions import garch_returns, formula_15, estimate_parameters_fast
from functions import legend_thick, remove_scientific_notation_from_vertical_axis

os.makedirs( 'plots', exist_ok = True )

In [ ]:
# Whether we compute the theoretical Var[SR] from the true values of the parameters or from estimated values.
# Using estimated values significantly slows down the computations 
TRUE_PARAMETERS = False

SEED = 0

In [ ]:
filename = "cache/parameters_daily_stock_returns.parquet"

if os.path.exists(filename):
        
    parameters = pd.read_parquet( filename )
    i1 = parameters['SR'] > 0
    i2 = parameters['T'] > 100
    i3 = parameters['denominator'] > 0

    np.random.seed(SEED)
    for _, row in parameters[i1 & i2 & i3][[
        'SR',
        'mu', 'sigma', 'skew', 'kurtosis', 
        'alpha', 'beta', 
        'tail_index', 'T', 'denominator', 
        #'nct_df', 'nct_nc', 'nct_loc', 'nct_scale',
        'skew_t_a', 'skew_t_b', 'skew_t_loc', 'skew_t_scale',
    ]].sample(100).iterrows():
        if row['skew'] < -.02 and row['kurtosis'] > 3.02 and row['SR'] > 0.02 and row['beta'] > .5 and row['alpha'] > .02 and row['T'] > 100:
            break

else: 
    row = {
        'SR': 0.07,
        'mu': 0.002,
        'sigma': 0.03,
        'alpha': 0.06,
        'beta': 0.87,
        'skew': -0.54,
        'kurtosis': 5.43,
        'skew_t_a': 1.7,
        'skew_t_b': 1.6,
        'skew_t_loc': -0.1,
        'skew_t_scale': 0.6,
        
    }

R = 1000  
Ts = np.unique( np.logspace( 1, 3, 100 ).astype(int) )  # Was: 100
@ray.remote
def f(row, Ts):
    SRs = []
    for T in Ts:
        var_theoretical = formula_15( SR = row['SR'], skew = row['skew'], kurtosis = row['kurtosis'], alpha = row['alpha'], beta = row['beta'], T = T )

        #innovations = scipy.stats.nct.rvs(row['nct_df'], row['nct_nc'], row['nct_loc'], row['nct_scale'], size=T)
        innovations = scipy.stats.jf_skew_t.rvs(row['skew_t_a'], row['skew_t_b'], row['skew_t_loc'], row['skew_t_scale'], size=T)
        
        y, _ = garch_returns( 
            size = T,
            mu = row['mu'], sigma = row['sigma'], alpha = row['alpha'], beta = row['beta'],
            innovations = innovations,
        )

        if not TRUE_PARAMETERS:
            parameters = estimate_parameters_fast( y )
            var_theoretical = formula_15( 
                SR = parameters.SR,
                skew = parameters.skew, 
                kurtosis = parameters.kurtosis, 
                alpha = parameters.alpha, 
                beta = parameters.beta, 
                T = T
            )
            
        SRs.append( { 
            'T': T,
            'var_theoretical': var_theoretical,
            'SR': y.mean() / y.std(),
        } )
    SRs = pd.DataFrame( SRs )
    return SRs

SRs = [ f.remote(row, Ts) for _ in range(R) ]
SRs = [ ray.get(u) for u in tqdm(SRs) ]
SRs = pd.concat( SRs )

In [ ]:
fig, ax = plt.subplots( figsize = ( 6,3 ), layout = 'constrained', dpi = 300 )
if TRUE_PARAMETERS:
    ax.plot( SRs.groupby('T')['var_theoretical'].mean(), color = 'tab:orange', label = "Theoretical" )
else:
    ax.scatter( 
        SRs['T'], 
        SRs['var_theoretical'], 
        color = 'tab:orange', 
        label = "Theoretical",
        s = 20,
        alpha = 1/255,
    )
ax.plot( SRs.groupby('T')['SR'].var(), color = 'tab:blue', label = "Sample" )
ax.set_xlabel('Number of observations')
ax.set_ylabel('Variance of SR')
ax.set_xscale('log')
if TRUE_PARAMETERS:
    legend_thick( ax )
else:
    leg = ax.legend()
    for lh in leg.legend_handles:
        lh.set_alpha(1)
#df_text = f"{row['nct_df']:.1f}" if row['nct_df'] < 100 else r"$\infty$"
title = ( 
    f"SR={row['SR']:.2f}"
    r"   $\alpha=$" f"{row['alpha']:.2f}" r"   $\beta=$" f"{row['beta']:.2f}" 
    f"   skew={row['skew']:.2f}   kurtosis={row['kurtosis']:.2f}"
    "\n"
    #f"nct(df={df_text}, nc={row['nct_nc']:.1f}, loc={row['nct_loc']:.1f}, scale={row['nct_scale']:.1f})"
    f"jf_skew_t(a={row['skew_t_a']:.1f}, b={row['skew_t_b']:.1f}, loc={row['skew_t_loc']:.1f}, scale={row['skew_t_scale']:.1f})"
)
ax.set_title( title )
ax.set_ylim( -.01, .2 )
ax.axhline( 0, color = 'black', linestyle = ':', linewidth = 1 )
filename = f"plots/plot_1__R={R}__T={len(Ts)}__{TRUE_PARAMETERS}__seed={SEED}"
plt.savefig( f"{filename}.png", dpi = 300 )
plt.show()

In [ ]:
tmp = pd.merge( 
    SRs.groupby('T')['SR'].var().reset_index(),
    SRs.set_index('T')[['var_theoretical']].reset_index(),
    on = 'T',
).set_index('T').rename( columns = { 'SR': 'var(SR)' } )
tmp['var(SR)/var(theoretical)'] = tmp['var(SR)'] / tmp['var_theoretical']
fig, ax = plt.subplots( figsize = ( 6,3 ), layout = 'constrained', dpi = 300 )
ax.scatter( tmp.index, tmp['var(SR)/var(theoretical)'], alpha = .01, s = 20 )
#tmp.reset_index().groupby('T')['var(SR)/var(theoretical)'].median().plot( color = 'white', linewidth = 1, ax = ax )
ax.axhline( 1, color = 'black', linestyle = ':', linewidth = 1 )
ax.set_ylim( 0, 2 )
ax.set_ylim(.1, 10); ax.set_yscale( 'log' )
ax.set_xscale( 'log' )
ax.set_xlabel( 'Number of observations' )
ax.set_ylabel( 'Variance of SR / Theoretical variance' )
ax.set_title( title )
plt.savefig( f"{filename}_relative_error.png", dpi = 300 )
plt.show()

In [ ]:
filename = "cache/parameters_daily_stock_returns.parquet"
if os.path.exists(filename):

    parameters = pd.read_parquet( "cache/parameters_daily_stock_returns.parquet" )
    i1 = parameters['SR'] > 0
    i2 = parameters['T'] > 100
    i3 = parameters['denominator'] > 0

    @ray.remote
    def f(id, row, R): 
        Ts = np.unique( np.logspace( 1, 3, 100 ).astype(int) ) 
        SRs = []
        for T in Ts:
            var_theoretical = formula_15( SR = row['SR'], skew = row['skew'], kurtosis = row['kurtosis'], alpha = row['alpha'], beta = row['beta'], T = T )
            for _ in range(R):
                #innovations = scipy.stats.nct.rvs(row['nct_df'], row['nct_nc'], row['nct_loc'], row['nct_scale'], size=T)
                innovations = scipy.stats.jf_skew_t.rvs(
                    row['skew_t_a'], row['skew_t_b'], row['skew_t_loc'], row['skew_t_scale'], 
                    size = T,
                )
                y, _, _ = garch_returns( 
                    size = T,
                    mu = row['mu'], sigma = row['sigma'], alpha = row['alpha'], beta = row['beta'],
                    innovations = innovations,
                )
                SRs.append( { 
                    'id': id,
                    'T': T,
                    'var_theoretical': var_theoretical,
                    'SR': y.mean() / y.std(),
                } )
        SRs = pd.DataFrame( SRs )
        return SRs

    R = 500
    N = 50
    SRs = []
    np.random.seed(SEED)
    for id, row in parameters[i1 & i2 & i3][[
        'SR',
        'mu', 'sigma', 'skew', 'kurtosis', 
        'alpha', 'beta', 
        'tail_index', 'T', 'denominator', 
        'nct_df', 'nct_nc', 'nct_loc', 'nct_scale',
        'skew_t_a', 'skew_t_b', 'skew_t_loc', 'skew_t_scale',
    ]].sample(N).iterrows():
        SRs.append( f.remote(id, row, R) )
    
    SRs = [ ray.get(u) for u in tqdm(SRs) ]
    SRs = pd.concat( SRs )

In [ ]:
if os.path.exists(filename):

    vars = SRs.groupby(['id', 'T']).agg(
        SR = ('SR', 'var'),
        var_theoretical = ('var_theoretical', 'mean'),
    ).reset_index()


    fig, ax = plt.subplots( figsize = ( 6, 3 ), layout = 'constrained', dpi = 300)
    ratio = ( 
        vars.pivot( index = 'T', columns = 'id', values = 'SR' ) /
        vars.pivot( index = 'T', columns = 'id', values = 'var_theoretical' )
    )
    for i in range(ratio.shape[1]):
        ax.scatter( 
            ratio.index, ratio.iloc[:, i],
            color = 'tab:blue',
            alpha = .1,
        )
    ax.axhline( 1, color = 'black', linestyle = ':', linewidth = 1 )
    ax.set_xscale('log')
    ax.set_yscale('log')

    #yticks = [.8, .9, 1, 1.1, 1.2, 1.3, 1.4, 1.5]
    #ax.set_yticks( yticks, yticks )
    remove_scientific_notation_from_vertical_axis( ax )

    ax.set_xlabel('Number of observations')
    ax.set_ylabel('Sample Variance of SR / Theoretical Variance of SR')
    filename = f"plots/plot_2__R={R}__N={N}__seed={SEED}"
    plt.savefig( f"{filename}.png", dpi = 300 )
    plt.show()